In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Create a sample customer dataset
df = pd.DataFrame({
    "AnnualIncome": [
        15, 18, 20, 22, 25, 28, 30, 32, 35, 38,
        40, 42, 45, 48, 50, 52, 55, 58, 60, 62,
        65, 68, 70, 72, 75, 78, 80, 85, 90, 95
    ],
    "SpendingScore": [
        39, 45, 15, 20, 35, 60, 65, 70, 75, 40,
        42, 80, 85, 20, 25, 30, 55, 60, 65, 15,
        18, 22, 35, 40, 75, 80, 85, 90, 25, 20
    ]
})

print("First 5 records:")
print(df.head())

# 2. Inspect data and missing values
print("\nDataset Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nSummary Statistics:")
print(df.describe())

# Visualize feature distributions
df.hist(figsize=(8, 4), bins=10)
plt.suptitle("Customer Feature Distributions")
plt.tight_layout()
plt.show()

# 3. Select and scale features
X = df[["AnnualIncome", "SpendingScore"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Elbow method
inertias = []

for k in range(1, 8):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.plot(range(1, 8), inertias, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(range(1, 8))
plt.grid(True)
plt.show()

# 5. Fit K-Means using a selected K
# Choose K after inspecting the elbow plot.
k = int(input("Enter the selected number of clusters K: "))

if k < 1 or k > 7:
    raise ValueError("K must be between 1 and 7.")

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Convert centroids back to original units
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

# 6. Plot clusters and centroids
plt.figure(figsize=(8, 5))

for cluster in sorted(df["Cluster"].unique()):
    cluster_data = df[df["Cluster"] == cluster]
    plt.scatter(
        cluster_data["AnnualIncome"],
        cluster_data["SpendingScore"],
        label=f"Cluster {cluster}"
    )

plt.scatter(
    centroids[:, 0], centroids[:, 1],
    marker="X", s=250, color="black", label="Centroids"
)

plt.xlabel("Annual Income (thousands)")
plt.ylabel("Spending Score")
plt.title("Customer Segmentation Using K-Means")
plt.legend()
plt.tight_layout()
plt.show()

# 7. Describe each customer segment
print("\nCluster Characteristics:")
summary = df.groupby("Cluster").agg(
    CustomerCount=("Cluster", "size"),
    AverageIncome=("AnnualIncome", "mean"),
    AverageSpending=("SpendingScore", "mean")
).round(2)

print(summary)
